<a href="https://colab.research.google.com/github/gj0210/CMP7239/blob/main/Ai%20dessertation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import re
import pandas as pd
import time

# 1. Define the Regex Patterns from your sources [cite: 280-287]
patterns = {
    "IPv4": r'\b(?:25[0-5]|2[0-4]\d|1?\d?\d)(?:\.(?:25[0-5]|2[0-4]\d|1?\d?\d)){3}\b',
    "Email": r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
    "CreditCard": r'\b(?:\d[ -]*){13,16}\b', # PDF suggests adding Luhn check later [cite: 283]
    "USB_Event": r'usb\s+\d+-\d+:\s+new\s+high-speed\s+USB\s+device\b',
    "SSH_Login": r'(Accepted|Failed) password for \s+(\w+)\s+from\s+(\d+\.\d+\.\d+\.\d+)'
}

def run_regex_pipeline(log_file):
    results = []
    start_time = time.time() # Objective 3: Track Runtime [cite: 93]

    with open(log_file, 'r') as f:
        for line_num, line in enumerate(f, 1):
            for artifact_type, pattern in patterns.items():
                matches = re.findall(pattern, line)
                for match in matches:
                    results.append({
                        "file": log_file,
                        "line": line_num,
                        "type": artifact_type,
                        "value": str(match)
                    })

    end_time = time.time()
    return pd.DataFrame(results), (end_time - start_time)

# Usage
df_regex, runtime_a = run_regex_pipeline("app.log")
display(df_regex)
print(f"Runtime: {runtime_a:.4f} seconds")

,file,line,type,value
0,app.log,1,Email,bob.jones@corp.local
1,app.log,14,USB_Event,usb 1-2: new high-speed USB device
2,app.log,17,Email,x_admin@sub.domain.org
3,app.log,21,USB_Event,usb 1-2: new high-speed USB device
4,app.log,22,Email,bob.jones@corp.local
5,app.log,23,Email,x_admin@sub.domain.org
6,app.log,33,USB_Event,usb 1-2: new high-speed USB device
7,app.log,34,Email,carol+support@ecom-shop.co.uk
8,app.log,39,USB_Event,usb 1-2: new high-speed USB device
9,app.log,41,Email,alice.smith@example.com


Runtime: 0.0064 seconds


In [6]:
import openai # Or your chosen LLM provider
import json

# Your Prompt Engineering Strategy [cite: 187, 246]
SYSTEM_PROMPT = """Extract all forensic artifacts (IPv4, Email, Credit Cards, USB events, Logins) from the logs.
Return ONLY a valid JSON list of objects with keys: 'type', 'value', 'line'."""

def run_genai_pipeline(log_chunk):
    # Log human effort spent on this prompt [cite: 202]
    start_time = time.time()

    response = openai.chat.completions.create(
        model="gpt-4o", # Or "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": log_chunk}
        ],
        response_format={ "type": "json_object" }
    )

    end_time = time.time()
    return json.loads(response.choices[0].message.content), (end_time - start_time)

To use the OpenAI API, you'll need an API key. If you don't already have one, create one on the [OpenAI website](https://platform.openai.com/account/api-keys).
In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `OPENAI_API_KEY`. Then pass the key to the SDK:

In [7]:
# Used to securely store your API key
from google.colab import userdata
import openai

from google.colab import userdata
userdata.get('OPENAI_API_KEY')

# Set your OpenAI API key from Colab secrets
try:
    openai.api_key = userdata.get('OPENAI_API_KEY')
    print("OpenAI API key loaded successfully.")
except userdata.SecretNotFoundError:
    print("Secret 'OPENAI_API_KEY' not found. Please add your OpenAI API key to Colab secrets.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

OpenAI API key loaded successfully.


Now, let's prepare a sample log chunk from `app.log` and run the `run_genai_pipeline` function.

In [8]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

with open("app.log", 'r') as f:
    log_chunk_sample = "".join(f.readlines()[:10]) # Read first 10 lines for a sample
import openai
from google.colab import userdata

retrieved_api_key = userdata.get('OPENAI_API_KEY')
openai.api_key = retrieved_api_key

print(f"API Key retrieved from Colab secrets (first 5 and last 3 chars): {retrieved_api_key[:5]}...{retrieved_api_key[-3:]}")

# Run the GenAI pipeline
df_genai, runtime_b = run_genai_pipeline(log_chunk_sample)
display(df_genai)
print(f"Runtime: {runtime_b:.4f} seconds")

FileNotFoundError: [Errno 2] No such file or directory: 'app.log'

In [9]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

'sk-...zfUA'

In [10]:
def calculate_metrics(extracted_df, ground_truth_df):
    # Merge extracted results with ground truth on value and line number
    matches = pd.merge(extracted_df, ground_truth_df, on=['value', 'line'], how='inner')

    tp = len(matches) # True Positives
    fp = len(extracted_df) - tp # False Positives [cite: 252]
    fn = len(ground_truth_df) - tp # False Negatives [cite: 253]

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {"Precision": precision, "Recall": recall, "F1": f1}